In [ ]:

from dataloader import create_dataloaders, load_noisy_dataset_by_task
from lora_model_alpha import LORAEngine
from influence import IFEngine

from tqdm import tqdm
import pickle as pkl
import torch
from torch.optim import AdamW
from torch.utils.data import DataLoader
from transformers import (
    AutoModelForSequenceClassification,
    
    
    
    
    get_linear_schedule_with_warmup,
    BitsAndBytesConfig,
    LlamaForCausalLM,
    LlamaTokenizer,
    AutoModelForCausalLM
)
import re

import numpy as np



In [ ]:
def extract_layer_number(s):
    match = re.search(r'\.(\d+)\.', s)
    return match.group(0) if match else None


base_path = "mistralai/Mistral-7B-v0.1"     #"mistralai/Mistral-7B-v0.1"    #"google/gemma-7b"    

base_model = AutoModelForCausalLM.from_pretrained(
    base_path,
    #quantization_config=quantization_config,
    load_in_4bit=True,
    torch_dtype=torch.bfloat16,
    offload_folder="offload",
    offload_state_dict=True,
    device_map='auto'
)
layers={}
for k,v in base_model.named_parameters():
    #print(k)
    result = extract_layer_number(k)
    layers[result]=0

layers=list(layers.keys())

layers.remove(None)

# IFs=[]

# for layer in layers:
#     with open(f'results_{layer}.pkl', 'rb') as f:
#         IF_pickle=pkl.load(f)
#     IFs.append(IF_pickle['influence']['proposed'].to_numpy().sum())

In [ ]:
# def scale_values_final(values, target_sum, exponent=1.5): #OLD IMPLEMENTATION
#     values = np.array(values, dtype=np.float64)  # Ensure numerical stability

#     # Invert values so less negative numbers remain smaller after transformation
#     inverted_values = np.max(values) - values  
    
#     # Avoid division by zero in case all values are identical
#     if np.all(inverted_values == 0):
#         inverted_values += 1e-6
    
#     scaled_values = np.power(inverted_values, exponent)

#     # Normalize to sum to target_sum
#     total_scaled = scaled_values.sum()
#     if total_scaled == 0:
#         raise ValueError("Scaling resulted in zero sum, check input values.")
    
#     # Convert scaled values into integer distribution while keeping sum == target_sum
#     scaled_fractions = (scaled_values / total_scaled) * target_sum
#     scaled_integers = np.round(scaled_fractions).astype(int)
    
#     # Ensure no values are zero (minimum value of 1)
#     scaled_integers = np.maximum(scaled_integers, 1)

#     # Adjust sum to match target_sum using the difference approach
#     while scaled_integers.sum() != target_sum:
#         difference = target_sum - scaled_integers.sum()
#         # print(scaled_integers.sum())

#         # Compute differences to decide where to adjust
#         adjustment_values = scaled_values - scaled_integers

#         if difference > 0:
#             idx = np.argmin(adjustment_values)  # Find smallest difference and increase it
#             scaled_integers[idx] += 1
#         else:
#             idx = np.argmax(adjustment_values)  # Find largest difference and decrease it
#             if scaled_integers[idx] > 1:
#                 scaled_integers[idx] -= 1  # Ensure minimum value remains 1

#     return scaled_integers

In [ ]:
def scale_values_final(values, target_sum, exponent=1.5):
    values = np.array(values, dtype=np.float64)
    n = len(values)

    # Invert values so smaller (less negative) ones get smaller weights
    inverted_values = np.max(values) - values
    if np.all(inverted_values == 0):
        inverted_values += 1e-6

    scaled_values = np.power(inverted_values, exponent)
    scaled_fractions = (scaled_values / scaled_values.sum()) * (target_sum - n)  # Subtract 1 per value

    # Floor and add the minimum value of 1
    floored = np.floor(scaled_fractions).astype(int) + 1
    remainder = target_sum - floored.sum()

    # Distribute remaining units to values with largest fractional parts
    fractional_parts = scaled_fractions - np.floor(scaled_fractions)
    top_indices = np.argsort(fractional_parts)[::-1]

    for i in range(remainder):
        floored[top_indices[i]] += 1

    return floored

In [ ]:
check=[2, 6, 6, 6, 6, 5, 7, 6, 8, 7, 6, 7, 6, 6, 6, 5,
 6, 6, 5, 4, 5, 5, 4, 2, 4, 5, 3, 4, 2, 3, 4, 3]
sum(check)

In [ ]:
# names=['mrpc', 'cola', 'openbook', 'text_scienceq','commonq']
# for name in names:
#     IFs = []
#     for layer in layers:
#         with open(f'results_{layer}_{name}.pkl', 'rb') as f:
#             IF_pickle=pkl.load(f)
#         IF_pickle_new=IF_pickle['influence']['identity'].to_numpy()
#         # print(IF_pickle_new)
#         summed_vector = np.sum(IF_pickle_new, axis=0, keepdims=True)
        
    
#         # print(IF_pickle['influence']['proposed'].to_numpy().sum())
#         # print(summed_vector.shape)
#         # print(np.where(summed_vector>0)[1])
#         remaining_values = np.delete(summed_vector, np.where(summed_vector>0)[1])
#         # print(np.sum(remaining_values))
#         # break
#         IFs.append(np.sum(summed_vector))
    
#     scaled_numbers_fixed = scale_values_final(IFs, target_sum=160, exponent=0.12)
#     print(name)
#     print(list(scaled_numbers_fixed))


In [ ]:
# import pickle as pkl

# with open(f'results_.25._mrpc.pkl', 'rb') as f:
#     IF_pickle=pkl.load(f)

In [ ]:
# IF_pickle

In [ ]:
# IF_pickle

In [ ]:
with open(f'outputs/layerIF_values/Mistral-7B-v0.1/results_Mistral-7B-v0.1_.4._commonq.pkl', 'rb') as f:
        IF_pickle=pkl.load(f)

In [ ]:
IF_pickle

In [ ]:
IF_pickle_new = IF_pickle['influence']['proposed'].to_numpy()
summed_vector = np.sum(IF_pickle_new, axis=0, keepdims=True)
summed_vector.shape

In [ ]:
summed_vector

In [ ]:
# b = np.sum(summed_vector[np.where(summed_vector>=0)[1]])
# a = np.sum(np.delete(summed_vector, np.where(summed_vector<0)[1]))

print(np.where(summed_vector>=0)[1])
print(np.where(summed_vector<0)[1])
print(f"summed_vector: {summed_vector.shape}")
print(np.sum(summed_vector[np.where(summed_vector<0)]))
print(np.sum(np.delete(summed_vector, np.where(summed_vector>=0)[1])))

In [ ]:
#All IF values 

names=['mrpc', 'cola', 'openbook', 'text_science_q_rebuttal','commonq']

for name in names:
    IFs = []
    for layer in layers:
        with open(f'outputs/layerIF_values/Mistral-7B-v0.1/results_Mistral-7B-v0.1_{layer}_{name}.pkl', 'rb') as f:
            IF_pickle=pkl.load(f)
        IF_pickle_new=IF_pickle['influence']['proposed'].to_numpy()
        # print(IF_pickle_new)
        summed_vector = np.sum(IF_pickle_new, axis=0, keepdims=True)
    
        # print(IF_pickle['influence']['proposed'].to_numpy().sum())
        # print(summed_vector.shape)
        # print(np.where(summed_vector>0)[1])
        # remaining_values = np.delete(summed_vector, np.where(summed_vector>0)[1])
        # print(np.sum(remaining_values))
        # break
        IFs.append(np.sum(summed_vector))
    print(f"IFs: {len(IFs)}, IFs[0]: {IFs[0].shape}") 
    
    scaled_numbers_fixed = scale_values_final(IFs, target_sum=160, exponent=2) #Can change target sum and exponent here
    print(name)
    scaled_numbers_fixed = list(scaled_numbers_fixed)
    print('expert numbers: ',','.join(map(str, scaled_numbers_fixed)))
    print('top_k:', ','.join(map(str, [1 if num==1 else 2 for num in scaled_numbers_fixed])))


In [ ]:
# #All_values

names=['mrpc', 'cola', 'openbook', 'text_science_q_rebuttal','commonq']

for name in names:
    IFs = []
    for layer in layers:
        with open(f'outputs/layerIF_values/Mistral-7B-v0.1/results_Mistral-7B-v0.1_{layer}_{name}.pkl', 'rb') as f:
            IF_pickle=pkl.load(f)
        IF_pickle_new=IF_pickle['influence']['proposed'].to_numpy()
        # print(IF_pickle_new)
        summed_vector = np.sum(IF_pickle_new, axis=0, keepdims=True)
    
        # print(IF_pickle['influence']['proposed'].to_numpy().sum())
        # print(summed_vector.shape)
        # print(np.where(summed_vector>0)[1])
        #remaining_values = np.delete(summed_vector, np.where(summed_vector>0)[1])
        # print(np.sum(remaining_values))
        # break
        IFs.append(np.sum(summed_vector))
    
    scaled_numbers_fixed = list(scale_values_final(IFs, 
                                                   target_sum=160, 
                                                   exponent=2))
    print(name, 'total:', sum(scaled_numbers_fixed))
    print(f'number_experts="{",".join(map(str, scaled_numbers_fixed))}"')
    print(f'top_k="{",".join(map(str, [1 if num==1 else 2 for num in scaled_numbers_fixed]))}"')


### All Negative IFs

In [ ]:
#Only positive IF Values

# IFs=[]
names=['mrpc', 'cola', 'openbook', 'text_science_q_rebuttal','commonq']  #, 'commonq'

for name in names:
    IFs = []
    for layer in layers:
        with open(f'outputs/layerIF_values/Mistral-7B-v0.1/results_Mistral-7B-v0.1_{layer}_{name}.pkl', 'rb') as f:
            IF_pickle=pkl.load(f)
        IF_pickle_new=IF_pickle['influence']['proposed'].to_numpy()
        summed_vector = np.sum(IF_pickle_new, axis=0, keepdims=True)
        # print(IF_pickle['influence']['proposed'].to_numpy().sum())
        # print(summed_vector.shape)
        # print(np.where(summed_vector>0)[1])
        remaining_values = np.delete(summed_vector, np.where(summed_vector>0)[1])
        # print(f"remaining values: {remaining_values}")
        # print(np.sum(remaining_values))
        # break
        IFs.append(np.sum(remaining_values))
    
    scaled_numbers_fixed = list(scale_values_final(IFs, target_sum=160, exponent=3)) #Can change target sum and exponent here
    # print(name)
    # print(list(scaled_numbers_fixed))
    print(name, 'total:', sum(scaled_numbers_fixed))
    print(f'number_experts="{",".join(map(str, scaled_numbers_fixed))}"')
    print(f'top_k="{",".join(map(str, [1 if num==1 else 2 for num in scaled_numbers_fixed]))}"')


In [ ]:
remaining_values

In [ ]:
# #Only positive gemma

# # IFs=[]
# #names=['mrpc', 'cola', 'openbook', 'text_scienceq','commonq']  #, 'commonq'
# names=['mrpc', 'cola', 'openbook', 'text_scienceq','commonq']
# for name in names:
#     IFs = []
#     for layer in layers:
#         with open(f'results_gemma_{layer}_{name}.pkl', 'rb') as f:
#             IF_pickle=pkl.load(f)
#         IF_pickle_new=IF_pickle['influence']['proposed'].to_numpy()
#         summed_vector = np.sum(IF_pickle_new, axis=0, keepdims=True)
#         # print(IF_pickle['influence']['proposed'].to_numpy().sum())
#         # print(summed_vector.shape)
#         # print(np.where(summed_vector>0)[1])
#         remaining_values = np.delete(summed_vector, np.where(summed_vector>0)[1])
#         # print(np.sum(remaining_values))
#         # break
#         IFs.append(np.sum(remaining_values))
    
#     scaled_numbers_fixed = scale_values_final(IFs, target_sum=160, exponent=2)
#     print(name)
#     print(list(scaled_numbers_fixed))


In [ ]:
np.sort(IFs)

In [ ]:
np.argsort(IFs)

In [ ]:
# #Only positive

# IFs=[]

# for layer in layers:
#     with open(f'results_{layer}_rte.pkl', 'rb') as f:
#         IF_pickle=pkl.load(f)
#     IF_pickle_new=IF_pickle['influence']['proposed'].to_numpy()
#     summed_vector = np.sum(IF_pickle_new, axis=0, keepdims=True)
#     # print(IF_pickle['influence']['proposed'].to_numpy().sum())
#     # print(summed_vector.shape)
#     # print(np.where(summed_vector>0)[1])
#     remaining_values = np.delete(summed_vector, np.where(summed_vector>0)[1])
#     print(np.sum(remaining_values))
#     # break
#     IFs.append(np.sum(remaining_values))

In [ ]:
np.argsort(np.abs(IFs))

In [ ]:
scale_values_final(IFs, 160)

In [ ]:
# Define the percentage of most negative values to keep (0.25, 0.50, or 0.75)
keep_ratio = 0.25 # Change this value as needed



names=['mrpc', 'cola', 'openbook', 'text_scienceq', 'commonq']
for name in names:
    IFs = []
    for layer in layers:
        with open(f'results_{layer}_{name}.pkl', 'rb') as f:
            IF_pickle = pkl.load(f)
        
        IF_pickle_new = IF_pickle['influence']['proposed'].to_numpy()
        summed_vector = np.sum(IF_pickle_new, axis=0)
        
        # Get the threshold for the most negative values
        num_values_to_keep = int(len(summed_vector) * keep_ratio)
        threshold = np.partition(summed_vector, num_values_to_keep)[num_values_to_keep]
        
        # Keep only the most negative values below the threshold
        remaining_values = summed_vector[summed_vector <= threshold]
        # print(remaining_values[0:10])
        # print(len(remaining_values))
        
        # print(np.sum(remaining_values))
        IFs.append(np.sum(remaining_values))



    # Scale the values ensuring no zero values and preventing errors
    scaled_numbers_fixed = scale_values_final(IFs, target_sum=160, exponent=2.5)
    print(name)
    print(list(scaled_numbers_fixed))

In [ ]:
# 2.5 beta, 25% threshold

# mrpc
# [4, 7, 8, 5, 6, 7, 7, 10, 11, 10, 8, 10, 6, 8, 7, 3, 7, 4, 3, 1, 6, 5, 1, 1, 5, 2, 1, 3, 1, 1, 1, 1]
# cola
# [1, 7, 9, 7, 8, 12, 11, 7, 10, 8, 8, 9, 6, 10, 7, 4, 7, 3, 2, 1, 3, 4, 2, 1, 6, 1, 1, 1, 1, 1, 1, 1]
# openbook
# [1, 7, 6, 12, 11, 7, 12, 6, 12, 10, 7, 6, 5, 5, 5, 7, 5, 4, 7, 3, 6, 3, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1]
# text_scienceq
# [1, 1, 1, 1, 1, 3, 4, 7, 3, 7, 6, 6, 6, 6, 8, 7, 7, 6, 6, 8, 7, 6, 6, 4, 5, 6, 6, 5, 4, 4, 8, 4]
# commonq
# [16, 12, 9, 10, 8, 6, 10, 5, 11, 8, 6, 8, 5, 6, 5, 2, 5, 4, 3, 1, 5, 4, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1]

In [ ]:
# 2.5 beta, 25% topk

# #mrpc
# 2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,1,2,2,1,1,2,2,1,2,1,1,1,1

# commonq
# 2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,1,2,2,1,1,1,2,1,1,1,1,1,1

# openbook
# 1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,1,1,1,2,2,2,1,1,1,1

# textscienceq
# 1,1,1,1,1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2

# cola
# 1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,1,2,2,2,1,2,1,1,1,1,1,1,1


In [ ]:
# 2.5 beta 50% threshold

# mrpc
# [1, 7, 7, 4, 5, 7, 8, 11, 8, 11, 9, 11, 6, 8, 7, 4, 7, 5, 3, 1, 6, 5, 1, 1, 6, 3, 1, 3, 1, 1, 1, 1]
# cola
# [1, 10, 9, 7, 8, 12, 10, 7, 10, 8, 8, 8, 6, 9, 7, 4, 7, 3, 2, 1, 3, 4, 2, 1, 6, 1, 1, 1, 1, 1, 1, 1]
# openbook
# [1, 6, 5, 10, 10, 7, 10, 6, 9, 9, 7, 6, 6, 6, 5, 7, 6, 5, 7, 4, 6, 4, 2, 1, 1, 3, 3, 3, 1, 1, 2, 1]
# text_scienceq
# [1, 1, 1, 1, 2, 2, 4, 6, 4, 6, 6, 6, 6, 6, 7, 7, 7, 7, 6, 3, 7, 7, 6, 5, 5, 6, 7, 6, 5, 5, 7, 5]
# commonq
# [2, 11, 9, 11, 8, 7, 11, 6, 8, 9, 8, 9, 6, 7, 6, 2, 7, 5, 4, 1, 6, 4, 2, 1, 1, 2, 1, 2, 1, 1, 1, 1]

In [ ]:
# 2.5 beta, 50% topk

# #mrpc
# 1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,1,2,2,1,1,2,2,1,2,1,1,1,1

# commonq
# 2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,1,2,2,2,1,1,2,1,2,1,1,1,1

# openbook
# 1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,1,1,1,2,2,2,1,1,1,1

# textscienceq
# 1,1,1,1,1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2

# cola
# 1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,1,2,2,2,1,2,1,1,1,1,1,1,1

In [ ]:
# 2.5 beta 75% threshold

# mrpc
# [1, 5, 6, 3, 5, 7, 8, 11, 7, 12, 10, 11, 7, 9, 7, 4, 7, 5, 4, 1, 6, 5, 1, 1, 6, 3, 1, 3, 1, 1, 1, 1]
# cola
# [1, 7, 10, 7, 9, 12, 10, 7, 9, 9, 9, 8, 6, 9, 6, 4, 7, 3, 3, 2, 3, 4, 2, 1, 5, 1, 1, 1, 1, 1, 1, 1]
# openbook
# [1, 6, 5, 9, 9, 6, 9, 7, 8, 9, 7, 6, 6, 6, 6, 7, 6, 5, 7, 4, 6, 5, 3, 1, 1, 3, 3, 3, 1, 2, 2, 1]
# text_scienceq
# [1, 1, 1, 1, 2, 2, 4, 6, 4, 6, 6, 6, 6, 6, 7, 6, 7, 7, 6, 4, 7, 6, 6, 5, 5, 6, 7, 6, 5, 6, 7, 5]
# commonq
# [1, 9, 7, 10, 8, 6, 11, 6, 9, 9, 8, 9, 6, 8, 7, 3, 7, 5, 5, 2, 6, 4, 2, 1, 1, 3, 1, 2, 1, 1, 1, 1]

In [ ]:
# 2.5 beta, 75% topk

#mrpc
# 1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,1,2,2,1,1,2,2,1,2,1,1,1,1

#cola
# 1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,1,2,1,1,1,1,1,1,1

#openbook
# 1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,1,1,2,2,2,1,2,2,1

#commonq
# 1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,1,1,2,1,2,1,1,1,1

#text_science_q
# 1,1,1,1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2

In [ ]:
#positive IF vals only

# mrpc
# 1,4,6,3,4,7,8,11,5,12,10,12,7,9,7,5,8,5,4,1,6,6,1,1,6,3,1,3,1,1,1,1
# cola
# 1,9,10,8,9,12,10,7,9,9,9,8,6,9,6,4,7,3,2,1,3,4,2,1,4,1,1,1,1,1,1,1
# openbook
# 1,5,5,8,7,5,8,7,8,5,7,7,7,6,6,6,6,6,7,5,7,5,3,2,2,4,3,4,2,2,3,1
# text_scienceq
# 1,1,2,1,2,2,4,6,4,6,6,6,6,6,7,6,7,7,6,3,7,6,6,5,5,6,7,6,5,6,7,5
# commonq
# 1,8,6,8,7,6,10,6,11,9,8,9,7,8,7,3,7,6,5,2,6,5,2,1,1,3,1,2,1,1,2,1

In [ ]:
#positive IF vals only topk

# mrpc
# 1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,1,2,2,1,1,2,2,1,2,1,1,1,1
# cola
# 1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,1,2,2,2,1,2,1,1,1,1,1,1,1
# openbook
# 1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,1
# text_scienceq
# 1,1,2,1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2
# commonq
# 1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,1,1,2,1,2,1,1,2,1

In [ ]:
# mrpc
# [1, 4, 5, 2, 4, 7, 8, 13, 13, 13, 11, 13, 7, 10, 7, 4, 8, 4, 3, 1, 5, 5, 1, 1, 5, 2, 1, 2]
# cola
# [1, 15, 10, 7, 9, 12, 11, 7, 9, 8, 9, 7, 6, 9, 6, 3, 7, 3, 2, 2, 3, 4, 2, 1, 4, 1, 1, 1]
# openbook
# [1, 5, 5, 8, 8, 6, 8, 7, 8, 9, 8, 7, 7, 6, 6, 6, 6, 6, 7, 5, 7, 5, 3, 2, 2, 4, 4, 4]
# text_scienceq
# [1, 2, 2, 1, 2, 3, 4, 6, 4, 7, 7, 6, 7, 7, 8, 7, 8, 8, 7, 8, 8, 7, 7, 6, 6, 6, 8, 7]
# commonq
# [1, 8, 6, 8, 7, 6, 10, 6, 13, 9, 8, 9, 7, 8, 7, 3, 7, 6, 5, 3, 6, 5, 2, 1, 2, 3, 2, 2]

In [ ]:
# Check if the same indexes have 1s in both rows1, 4, 5, 2, 4, 7, 8, 13, 13, 13, 11, 13, 7, 10, 7, 4, 8, 4, 3, 1, 5, 5, 1, 1, 5, 2, 1, 2
number_experts = [5,5,5,5,5,5,5,5,5,5,5,5,5,5,6,5,5,1,6,6,5,5,5,5,5,6,5,5,5,5,5,5]
# top_k = [1, 2, 2, 1, 2, 3, 4, 6, 4, 7, 7, 6, 7, 7, 81,2,2,1,2,3,4,6,4,7,7,6,7,7,8,7,8,8,7,8,8,7,7,6,6,6,8,7, 7, 8, 8, 7, 8, 8, 7, 7, 6, 6, 6, 8, 7]

# Find indexes where top_k has 1s
indexes_with_ones = [i for i, val in enumerate(number_experts) if val == 1]

# Check if the same indexes have 1s in number_experts
top_k=[]
for i in range(len(number_experts)):
    if number_experts[i]==1:
        top_k.append(1)
    else:
        top_k.append(2)
print(top_k)



In [ ]:
scaled_numbers_fixed  #rte

In [ ]:
scaled_numbers_fixed  #mrpc

In [ ]:
scaled_numbers_fixed #cola

In [ ]:
scaled_numbers_fixed  #openbook

In [ ]:
scaled_numbers_fixed  #commonq

In [ ]:
scaled_numbers_fixed  #text_science_q

In [ ]:
scaled_numbers_fixed  #cola

In [ ]:
scaled_numbers_fixed   #openbook

In [ ]:
scaled_numbers_fixed  #rte

In [ ]:
scaled_numbers_fixed  #mrpc

In [ ]:
scaled_numbers_fixed  #commonq_new

In [ ]:
scaled_numbers_fixed #text_science_q_new

In [ ]:
# expert number:  1,3,5,4,5,5,4,4,3,4,3,2,2,3,3,4,9,4,7,7,7,7,7,7,9,7,6,8,6,7,4,3   alphalora
# top_k:  1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2

In [ ]:
scaled_numbers_fixed ##text_science_q-10

# expert number:  1,3,5,4,5,5,4,4,3,4,3,2,2,3,3,4,9,4,7,7,7,7,7,7,9,7,6,8,6,7,4,3   alphalora

In [ ]:
scaled_numbers_fixed ##rte-100/10

In [ ]:
scaled_numbers_fixed ###openbook-100/10

In [ ]:
scaled_numbers_fixed ###commonq-100/10

In [ ]:
scaled_numbers_fixed ###cola-100/10

In [ ]:
scaled_numbers_fixed.sum()

In [ ]:
import os
import re

# Directory containing the files
directory = "/nas02/Hadi/Model-Selection-IF/alphalora/DataInf/src"

# Regex pattern to match files (N can be 0-32, X can be specified values or ".")
pattern = re.compile(r"results_\.(\d{1,2})\._(cola|rte|openbook|commonq|text_scienceq|\.)\.pkl")

# Define new naming format (modify as needed)
def new_name(old_name, match):
    N = match.group(1)  # Extract the number (0-32)
    X = match.group(2)  # Extract the dataset name or "."
    
    # Example renaming: Change "results_.N._X.pkl" to "renamed_N_X.pkl"
    new_filename = f"renamed_{N}_{X}.pkl"  # Replace "." with "dot" for clarity
    return new_filename

# Iterate through files and rename them
for filename in os.listdir(directory):
    match = pattern.match(filename)
    if match:
        new_filename = new_name(filename, match)
        old_path = os.path.join(directory, filename)
        new_path = os.path.join(directory, new_filename)

        os.rename(old_path, new_path)
        print(f"Renamed: {filename} -> {new_filename}")


In [ ]:
import os
import re

# Directory containing the files
directory = "/nas02/Hadi/Model-Selection-IF/alphalora/DataInf/src"

# Regex pattern to match files (N can be 0-32)
pattern = re.compile(r"results_\.(\d{1,2})\.\.pkl")

# Function to generate new filename
def new_name(old_name, match):
    N = match.group(1)  # Extract the number (0-32)
    return f"renamed_{N}_mrpc.pkl"  # New format

# Iterate through files and rename them
for filename in os.listdir(directory):
    match = pattern.match(filename)
    if match:
        new_filename = new_name(filename, match)
        old_path = os.path.join(directory, filename)
        new_path = os.path.join(directory, new_filename)

        os.rename(old_path, new_path)
        print(f"Renamed: {filename} -> {new_filename}")


In [ ]:
with open(f'results_.31._commonq.pkl', 'rb') as f:
    IF_pickle=pkl.load(f)

In [ ]:
IF_pickle.keys()

In [ ]:
IF_pickle['influence']['proposed']